In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import time
import serial
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Instruments
from Instruments.GX_271.gilson_ethernet import GilsonEthernet
from Instruments.GX_271.tray import Tray
from Instruments.GX_271.rack import Rack_209, Rack_3dp
from Instruments.VICI_valves.dim import DIM
from Instruments.VICI_valves.fsm import FSM
from Instruments.Syr_pumps.HB_syr_pump import HBElite
from Instruments.Knauer.knauer_pump_azura import KnauerPumpAzura

# Ecosystems
from Ecosystems.reaction_segment_generation import RSG
from Ecosystems.segmentation import Segmentation

# Logger
from Core.logging import flow_logger as logger

# Experiment framework
from Core.experiment_context import ExperimentManager
from Core.experiment_builder import ExperimentBuilder
from Core.experiment_validator import ExperimentValidator
from Core.experiment_compiler import ExperimentCompiler
from Core.experiment_intent import ExperimentIntent

In [ ]:
# --- Syringe pump ---
syr_pump = HBElite("COM10")
syr_pump.connect()

# --- DIM ---
dim = DIM("COM5")
dim.connect()

# --- FSM ---
fsm = FSM("COM2")
fsm.connect()

# --- Carrier pump ---
k_pump = KnauerPumpAzura("COM4")
k_pump.connect()

# --- Gilson ---
#--- Construct the tray ---
tray = Tray()

# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.101", admin_port=50185, tray=tray)

# --- Tell gilson instance about the DIM ---
g.dim = dim

# --- Start logging (declare this run belongs to the experiment "xxxxx") ---
logger.start_experiment("Segmentation_testing")

# --- Instantiate modules (racks, wash stations, etc.) (these know internal geometry) ---
rack1 = Rack_209()  
rack2 = Rack_3dp()

# --- Register modules on the tray with global offsets, previously handled by each module in the tray ---
tray.add_module(
    slot=1,
    name="rack1",
    module=rack1,
    x_offset=155.5,
    y_offset=10,
    x_min=145,
    x_max=250,
    y_min=2,
    y_max=292
)

tray.add_module(
    slot=2,
    name="rack2",
    module=rack2,
    x_offset=319,
    y_offset=39,
    x_min=275,
    x_max=370,
    y_min=2,
    y_max=292
) 

tray.add_module(
    slot=3,
    name="dim",
    module=dim,
    x_offset=9,     #These values are to be changed
    y_offset=104,
    x_min=0,
    x_max=25,
    y_min=75,
    y_max=120
) 

# --- Ecosystems ---
rsg = RSG(gilson=g, pump=syr_pump, dim=dim)

seg = Segmentation(
    dim=dim,
    fsm=fsm,
    carrier_pump=k_pump,
    rsg=rsg
)

print("Hardware connected.")

In [ ]:
inventory = Inventory()

# Clear + reassign (optional but safer for now)
inventory.assign(
    module="rack1",
    vial=1,
    name="BnOH_stock",
    concentration_M=0.0811,   # adjust if known
    solvent="MeCN",
    current_volume_uL=1500,
    min_safe_volume_uL=200
)

inventory.assign(
    module="rack2",
    vial=1,
    name="MeCN",
    concentration_M=None,
    solvent="MeCN",
    current_volume_uL=5000,
    min_safe_volume_uL=500
)

In [ ]:
from Core.experiment_intent import ExperimentIntent

intent = ExperimentIntent(
    experiment_id="VB-1-13",
    description="10x repeated 100 uL BnOH_stock slug in MeCN"
)

intent.add_block(
    name="BnOH single slug repeats",
    components=["BnOH_stock"],
    ratios=[[100] for _ in range(10)],  # 10 identical slugs
    total_volume_uL=100.0
)

intent.summary()

In [ ]:
compiler = ExperimentCompiler(inventory)

compiled_df = compiler.compile(intent)

compiled_df.head()

In [ ]:
compiled_df.groupby("slug_id").size()

In [ ]:
builder = ExperimentBuilder(inventory=inventory)

result = builder.build_and_create(
    experiment_id="VB-1-13",
    rows=compiled_df,
    description=intent.description,
    global_conditions={
        "flowrate_ul_min": 1000,
        "gas_prime_s": 20
    },
    overwrite=True
)

print(result["plan_path"])
pd.DataFrame(result["summary"])

In [ ]:
manager = ExperimentManager()

manager.load_experiment("VB-1-13")
manager.status()

In [ ]:
manager.precondition_reactor(seg, carrier_flowrate_ul_min=1000, stabilisation_time_s=60)

manager.prepare_rsg(
    seg,
    air_gap=20.0,
    rate_wdr=0.10
)

manager.arm_experiment()

In [ ]:
preview = manager.preview()
pd.DataFrame(preview)

In [ ]:
results = manager.execute_experiment(seg)
results